# **Comprehensive Guide to RAG Architectures**

This notebook explores different patterns in Retrieval-Augmented Generation (RAG) architectures:
1. Basic Naive RAG Pipeline
2. Hybrid Search RAG (Vector + Keyword)
3. RAG with Reranking (Cross-Encoder)
4. Query Expansion / Transformative RAG
5. Agentic RAG
6. GraphRAG Architecture

**Prerequisites:** Install dependencies before running:
```bash
pip install langchain langchain-community langchain-chroma sentence-transformers faiss-cpu pandas matplotlib chromadb python-dotenv
```

## **Table of Contents**
1. [Basic Naive RAG](#1-basic-naive-rag)
2. [Hybrid Search RAG](#2-hybrid-search-rag)
3. [RAG with Reranking](#3-rag-with-reranking)
4. [Query Expansion RAG](#4-query-expansion-rag)
5. [Agentic RAG](#5-agentic-rag)
6. [GraphRAG Architecture](#6-graghg-architecture)

---

In [ ]:
import os
import time
from typing import List, Dict, Optional

os.environ["LANGCHAIN_API_KEY"] = "demo_key" # Placeholder for demo
# Configure ChromaDB embedding store location for this notebook environment
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS

## **1. Basic Naive RAG Pipeline**

**Description:** The simplest RAG architecture. It takes a user query, converts it to an embedding, searches the vector database for top-k documents, and then feeds them to an LLM for generation.

**Pros:** Simple to implement, low latency.
**Cons:** No query rewriting, no reranking. May retrieve irrelevant context if query is not exact match.

In [ ]:
from langchain_community.llms import FakeListLLM  # Demo LLM (In practice, use OpenAI/HuggingFace)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.embeddings import HuggingFaceEmbeddings

# Simulated embedding model - in production use 'HuggingFaceEmbeddings' or Azure/SendGrid/VertexAI
from langchain_huggingface import HuggingFaceEmbeddings

# Define embeddings using a local HuggingFace model (for demo purposes)
# Using sentence-transformers/all-MiniLM-L6-v2 for lightweight demo
try:
    from langchain_community.embeddings import HuggingFaceEmbeddings
    embedding_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
except:
    # Mock embeddings for quick notebook run if offline
    class FakeEmbeddings:
        def __init__(self):
            pass
        def embed_documents(self, texts: List[str]) -> List[List[float]]:
            return [[0.1]*768 for _ in texts]
        def embed_query(self, text: str) -> List[float]:
            return [0.1]*768

    embedding_model = FakeEmbeddings()

# Simulated documents (in practice these would be loaded from PDFs/Text files)
documents_content = """
### Company Information
The company was founded in 2015 and specializes in AI solutions.
Our headquarters is located in New York City. We have over 500 employees worldwide.

### Product Features
Product A provides real-time analytics for enterprise customers.
Product B includes advanced security protocols with zero-trust architecture.
Both products support cloud and on-premise deployments.

### Contact Information
Customer support can be reached at support@company.com or +1-800-555-0199.
Our main website is www.example-company.com
""".strip()

# Create documents list
documents = [Document(page_content=documents_content)]

# Split documents into chunks (simulated)
# In real notebook, use text.splitter:
from langchain_core.documents import Document

docs_list = []
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
for doc in documents:
    text_chunks = text_splitter.split_text(doc.page_content)
    docs_list.extend([Document(page_content=text) for text in text_chunks])

# Create vector store (in-memory FAISS for demo)
from langchain_core.embeddings import Embeddings

class MockEmbedding:
    def embed_documents(self, texts): return [[0.1]*768] * len(texts)
    def embed_query(self, text): return [0.1]*768

embeddings = MockEmbedding()  # Replace with real embedding model
vector_store = FAISS.from_documents(docs_list, embeddings)

# Create RAG Chain
from langchain_core.prompts import ChatPromptTemplate

prompt_template = """You are a helpful assistant. Use the following context to answer the question. If the answer is not in the context, just say 'I'm sorry, I don't have enough information.'
Context: {context}
Question: {question}
Answer:"""

prompt = ChatPromptTemplate.from_template(prompt_template)

# Demo LLM (replace with actual model like OpenAI, Anthropic, or HuggingFace)
from langchain_core.language_models import BaseChatModel

class MockLLM:
    def __init__(self):
        pass
    
    def generate(self, prompts):
        from langchain.schema import LLMResult
        return LLMResult(generations=[[{"text": "Answer based on context (simulated)."}]])

llm = MockLLM()
llm_chain = prompt | llm.generate

# Basic RAG Function
def basic_rag(query: str, k=3):
    retrieved_docs = vector_store.similarity_search_with_score(query, k=k)
    context_text = "\n\n".join([doc[0].page_content for doc in retrieved_docs])
    
    response = llm_chain.invoke({"question": query, "context": context_text})
    return response

# Example Call to Basic RAG
try:
    response_basic = basic_rag("What is the company's email support?")
    print(response_basic)
except Exception as e:
    print(f"Basic RAG demo note: {e}")

---

## **2. Hybrid Search RAG (Vector + Keyword)**

**Description:** Combines vector embeddings for semantic similarity and BM25 (term frequency-inverse document frequency) for keyword matching. Improves retrieval when users ask questions using different words than the source documents.

**Pros:** Captures both conceptual and literal search relevance.
**Cons:** Requires two retrieval engines and merging logic.

In [ ]:
# Hybrid Search Architecture Implementation
class VectorKeywordHybrid:
    def __init__(self, embeddings):
        self.embeddings = embeddings
    
    def hybrid_search(self, query: str, k=5) -> List[Document]:
        # 1. Retrieve via Vector
        vector_docs = []
        try:
            vectors_results = FAISS.from_documents(docs_list, self.embeddings).similarity_search_with_score(query, k=k)
            for doc, score in vectors_results[:k]:
                vector_docs.append(doc)
        except Exception as e:
            print(f"Vector search error (using demo fallback): {e}")
        
        # 2. Retrieve via BM25 (Keyword search - simulated here with text matching)
        keyword_docs = []
        try:
            from rank_bm25 import BM25Okapi
            tokenized_texts = [doc.page_content.replace(" ", " ") for doc in docs_list]
            bm25 = BM25Okapi(tokenized_texts)
            tokens = query.split()
            
            scores = bm25.get_scores(tokens)
            best_k_idx = sorted(range(len(scores)), key=lambda x: scores[x], reverse=True)[:k]
            
            for idx in best_k_idx:
                if hasattr(docs_list[idx], 'page_content'):
                    keyword_docs.append(Document(page_content=docs_list[idx].page_content))
        except Exception:
            print("Keyword search simulated as fallback.")

        # 3. Merge and deduplicate
        all_docs = {doc.page_content[:50] + "...": doc for doc in vector_docs} | {doc.page_content[:50] + "...": doc for doc in keyword_docs}
        return list(all_docs.values())[:k*2]

# Example of hybrid search (mocked)
hybrid_rag = VectorKeywordHybrid(embeddings)
print("Hybrid RAG ready to use with real BM25 engine")

---

## **3. RAG with Reranking (Cross-Encoder)**

**Description:** After retrieving initial top-k vectors, a cross-encoder model re-scores the results based on their semantic relevance to the query. This filters out less relevant documents before passing to LLM generation.

**Pros:** Higher precision. Uses dense text similarity models like Cohere Embeddings or Cross-Encoder (BGE).
**Cons:** Adds a slight computational cost and latency.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
try:
    reranker = None  # Replace with cross-encoder model
except Exception:
    print("Reranking not implemented in this demo environment.")
    print("Use Cohere Rerank or BGE Cross-Encoder in production.")

# Pseudo-code for reranking logic:
def apply_reranker(query, initial_docs, k):
    # 1. Retriever returns top-k (e.g., 10)
    # 2. Reranker cross-encodes query + doc pair
    # 3. Returns new ordered list based on scores
    
    # Example: Cohere's rerank command via API
    from langchain_cohere import CohereRerank
    reranker = CohereRerank(model="rerank-english-v2.0")
    ranked_docs = reranker.invoke(query, initial_docs[:10], k=k)
    return ranked_docs

---

## **4. Query Expansion / Transformative RAG**

**Description:** Instead of feeding the raw query directly to the retriever, an LLM rewrites or expands the query (e.g., splitting a multi-part question into separate sub-queries). Also includes HyDE (Hypothetical Document Embeddings).

**Pros:** Handles complex, ambiguous, or multi-intent questions better.
**Cons:** Requires additional LLM tokens and latency before retrieval.

In [ ]:
class QueryExpansionRAG:
    def __init__(self):
        self.query_expander_llm = None  # Use an LLM like OpenAI or HuggingFace
    
    def expand_query(self, query: str) -> List[str]:
        if not self.query_expander_llm:
            return [query]
        
        prompt = f"Rewrite this query for better search retrieval. Break into multiple parts if needed.\nQuery: {query}\nExpanded Queries:"
        response = self.query_expander_llm.invoke(prompt).content.strip()
        # Split and normalize expanded queries
        sub_queries = [q.strip().strip("'") for q in response.split('\n')]
        return sub_queries
    
    def retrieve_and_combine(self, sub_queries: List[str], k=5) -> str:
        all_retrieved_contexts = ""
        for sub_query in sub_queries:
            docs = FAISS.from_documents(docs_list, embeddings).similarity_search_with_score(sub_query, k=k)
            for doc, _ in docs:
                all_retrieved_contexts += f"\n{doc.page_content}\n"
        
        return all_retrieved_contexts

---

## **5. Agentic RAG (Advanced)**

**Description:** Uses an LLM "agent" to dynamically plan the retrieval strategy based on user questions. The agent can decide whether to search, retrieve, use a tool (e.g., Google Search), or call APIs.

**Pros:** Handles multi-hop reasoning and complex workflows autonomously.
**Cons:** More complex orchestration, higher token costs, potential for loops.

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

class AgentState(TypedDict):
    query: str
    plan: str
    retrieved_context: str
    final_answer: str

def create_agent_graph():
    """
    Create a simple agent graph for RAG with planning capabilities.
    This represents an Agentic workflow where LLM decides how to retrieve info.
    """
    
    # Define nodes (simulating different decision points)
    def plan_node(state):
        query = state.get('query', '')
        # Simulated planning logic - in production this would be an actual LLM call
        planned_queries = [query]  # Simplified for demo
        return {"plan": f"Plan to search: {planned_queries}"}
    
    def retrieve_node(state):
        # Call your vector/retriever system based on the plan
        return {"retrieved_context": "Retrieved context documents..."}
    
    def generate_node(state):
        # Generate final answer using retrieved context
        return {"final_answer": "Final answer generated from context."}
    
    # Build graph (simplified)
    # In production use: from langgraph.graph import StateGraph, END
    
    print("Agent Graph Structure defined")
    print("Flow: Query -> Plan -> Retrieve -> Generate")
    

**Example Agentic Workflow Execution**

def run_agent_rag(query):
    """Simulates agentic RAG execution"""
    print(f"User Query: {query}")
    
    # Step 1: Agent plans approach
    print("Agent planning retrieval strategy...")
    plan_result = {"approach": "hybrid_search"}
    
    # Step 2: Execute retrieval
    print("Executing retrieval based on plan...")
    retrieved = vector_store.similarity_search(query, k=5)
    
    # Step 3: Generate answer
    print("Generating final response...")
    context = "\n".join([doc.page_content for doc in retrieved])
    
    return f"Answer generated from {len(retrieved)} documents."

---

## **6. GraphRAG Architecture**

**Description:** Combines knowledge graphs (structured relationships) with vector search. Instead of just storing embeddings, we extract entities and build a graph where nodes represent people/organizations/concepts and edges represent 
relationships. This allows answering relationship-based questions better.

**Pros:** Excellent for entity relationship questions, better at multi-hop reasoning across connected data.
**Cons:** More complex setup, requires graph database integration.

In [ ]:
import networkx as nx  # For graph visualization in notebook

def create_knowledge_graph(documents):
    """
    Create a simplified knowledge graph from documents.
    Extracts entities and their relationships.
    """
    # In production, would use Neo4j or NebulaGraph for persistent storage
    # This is a conceptual demonstration
    
    G = nx.Graph()
    
    # Extract sample entities (simulated)
    sample_entities = [
        ("Company A", "New York"),
        ("Company B", "San Francisco"), 
        ("AI Researcher", "Tech Lead"),
        ("Product X", "Analytics Platform")
    ]
    
    # Add nodes
    for entity, property_val in sample_entities:
        G.add_node(entity)
    
    # Add edges (relationships)
    relations = [
        ("Company A", "located_in", "New York"),
        ("AI Researcher", "works_at", "Company A"),
        ("Product X", "developed_by", "Tech Lead")
    ]
    
    for node1, edge_type, node2 in relations:
        G.add_edge(node1, node2)
    
    return G

**GraphRAG Query Flow**

def graghg_query(query):
    """
    GraphRAG query execution:
    1. Convert query to embeddings + keywords
    2. Search vector DB for related documents
    3. Traverse knowledge graph for entity relationships
    4. Combine results for generation
    """
    print("GraphRAG Query Processing:")
    print("  1. Embedding query...")
    
    print("  2. Retrieving from vector store...")
    
    print("  3. Traversing knowledge graph relationships...")
    
    print("  4. Generating answer with relationship context...")
    
    return "Answer with entity relationships and provenance."

---

## **7. Multi-Stage / Tiered RAG**

**Description:** Implements a multi-tier approach where simple queries go to a fast cache (e.g., vector index), while complex questions trigger deeper analysis (graph search, agent planning, external tools).

**Pros:** Optimizes performance by routing different query types appropriately.
**Cons:** Requires more infrastructure and careful routing logic.

In [ ]:
class MultiStageRAG:
    def __init__(self):
        self.cache = {}  # Fast lookup cache
        self.vector_store = vector_store
        self.graph_engine = nx.Graph()  # Simplified graph reference
    
    def route_query(self, query):
        """Decides which stage to use based on query complexity"""
        
        # Simple keyword matching first
        if any(keyword in query for keyword in ["who", "when", "where", "what"]):
            return "vector_search"
        
        # Check for relationship questions (multi-hop)
        if any(word in query for word in ["relationship", "between", "from", "to"]):
            return "graph_search"
        
        # Complex or open-ended queries
        return "agent_or_rerank"

---

## **Comparison Table of All RAG Architectures**

This table summarizes all architectures presented in this notebook.

In [ ]:
import pandas as pd

comparison_df = {
    'Architecture': ['Basic Naive RAG', 'Hybrid Search RAG', 
                    'RAG with Reranking', 'Query Expansion RAG', 
                    'Agentic RAG', 'GraphRAG'],
    
    'Complexity': ['Low', 'Medium', 'Medium-High', 'High', 'Very High', 'High'],
    'Latency': ['Fastest', 'Moderate', 'Slower (rerank)', 'Slower (expand)', 
                'Slowest (agent loops)', 'Moderate (graph)'],
    
    'Accuracy': ['Basic', 'Better than Basic', 'High Precision', 
                'Context-Aware', 'Dynamic', 'Relationship-Focused'],
    
    'Best Use Case': ['Simple Q&A', 'Semantic + Keyword', 
                    'High Precision Required', 'Complex Questions', 
                    'Multi-step Reasoning', 'Entity Relationship Queries'],
                    
    'Key Components': ['Vector DB + LLM', 'Vector + BM25', 
                     'Cross-Encoder Reranker', 'LLM Query Rewriter', 
                     'Agent Planner', 'Knowledge Graph + Vector DB']
}

comparison_df = pd.DataFrame(comparison_df)

---

## **Final Recommendations**

Choose your architecture based on your needs:

- **Start with Basic RAG:** If you want a simple implementation.
- **Upgrade to Hybrid Search:** When semantic and keyword search both matter.
- **Add Reranking:** For high-stakes applications requiring precision.
- **Use Query Expansion:** To handle complex user queries better.
- **Consider Agentic RAG:** When you need autonomous decision-making.
- **GraphRAG:** Essential for domain knowledge with relationships (companies, people, products).

---

## **Additional Resources**

1. LangChain: https://js.langchain.com/
2. LlamaIndex: https://docs.llamaindex.ai/
3. Microsoft GraphRAG Blog: https://www.microsoft.com/en-us/research/blog/gragh-a-graph-based-approach-for-retreival-augmented-generation

print("\n" + "="*80)
print("COMPREHENSIVE RAG ARCHITECTURES NOTEBOOK COMPLETED")
print("="*80)